In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table
import pandas as pd


class ADXClient:
    """Thin wrapper around KustoClient with switchable authentication."""

    def __init__(self, cluster: str, interactive_login: bool = False):
        if not cluster:
            raise ValueError("Pass a cluster URL when creating an ADXClient.")
        self.cluster = cluster
        self.interactive_login = interactive_login
        self.credential = self._create_credential()
        self.client = self._create_client()

    def _create_credential(self):
        if self.interactive_login:
            return InteractiveBrowserCredential()
        return DefaultAzureCredential()

    def _create_client(self) -> KustoClient:
        kcsb = KustoConnectionStringBuilder.with_azure_token_credential(
            self.cluster, self.credential
        )
        return KustoClient(kcsb)

    def set_interactive_login(self, interactive_login: bool) -> None:
        self.interactive_login = interactive_login
        self.credential = self._create_credential()
        self.client = self._create_client()

    def perform_query(
        self,
        query: str | None = None,
        table: str | None = None,
        database: str | None = None,
        take_limit: int | None = None,
    ) -> pd.DataFrame | None:
        """Perform a raw KQL query or a table query and return a pandas DataFrame."""
        if database is None:
            raise ValueError("Pass database to perform_query.")
        if query is None and table is None:
            raise ValueError("Pass either query or table to perform_query.")
        if query is not None:
            resolved_query = query
        elif take_limit is None:
            resolved_query = table
        else:
            resolved_query = f"{table} | take {take_limit}"

        try:
            response = self.client.execute(database, resolved_query)
            return dataframe_from_result_table(response.primary_results[0])
        except Exception as exc:
            print(f"Query failed with error: {exc}")
            return None